In [20]:
!pip install "langchain==0.3.25" "langchain-community==0.3.24" "langchain-text-splitters==0.3.8" "langchain-google-genai==2.1.5" faiss-cpu sentence-transformers pypdf -q

In [21]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [22]:
from google.colab import drive
drive.mount('/content/drive')

import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

DATA_PATH = "/content/drive/MyDrive/my_personal_data"
os.makedirs(DATA_PATH, exist_ok=True)  # creates the folder if it doesn't exist yet

documents = []
documents += DirectoryLoader(DATA_PATH, glob="**/*.txt", loader_cls=TextLoader).load()
documents += DirectoryLoader(DATA_PATH, glob="**/*.pdf", loader_cls=PyPDFLoader).load()

print(f"Loaded {len(documents)} documents")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded 10 documents


In [23]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

Split into 10 chunks


In [24]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("/content/drive/MyDrive/my_rag_index")
print("Vector store saved.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector store saved.


In [25]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0.3)

prompt_template = """You are a helpful assistant that answers questions using only the context below, which is personal information about the user. If the answer isn't in the context, say you don't know — don't make things up.

Context:
{context}

Question: {question}
Answer:"""

prompt = PromptTemplate(template=prompt_template, input_variables=["context", "question"])
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

In [26]:
response = qa_chain.invoke({"query": "what is aI"})
print(response["result"])

Based on the provided context, AI stands for Artificial Intelligence. It is described as an educator's "most powerful assistant" and a "partner in the classroom" that reshapes learning, personalises student journeys, and automates routine administrative tasks (like grading and scheduling) to augment human intelligence. The context does not provide a broader or technical definition of AI beyond its role in education.
